In [1]:
%pip install polars s3fs fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 38.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 170.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import s3fs
import polars as pl
import os

In [3]:
bucket = "backblaze-afr"
prefix = "processed/drivestats"

print("1. Connecting to S3 to find files...")
fs = s3fs.S3FileSystem()
all_files = fs.glob(f"{bucket}/{prefix}/**/*.parquet")

sample_files = all_files[:10] + all_files[-10:]

master_schema = {}
print("2. Skimming metadata to build the Master Schema...")

for file in sample_files:
    file_schema = pl.scan_parquet(
        f"s3://{file}",
        storage_options={"region": "us-west-1"}
    ).collect_schema()

    master_schema.update(dict(file_schema))

print("Overriding problematic data types in the Master Schema...")

if "failure" in master_schema:
    master_schema["failure"] = pl.Int64

print(f"Success! Discovered {len(master_schema)} unique columns across the timeline.\n")

print("3. Scanning the entire dataset with the unified schema...")

lazy_df = pl.scan_parquet(
    f"s3://{bucket}/{prefix}/**/*.parquet",
    storage_options={"region": "us-west-1"},
    schema=master_schema,
    missing_columns="insert",
    extra_columns="ignore",
    cast_options=pl.ScanCastOptions(integer_cast='upcast')
)

smart_cols = [col for col in master_schema.keys() if "smart" in col]
lazy_df = lazy_df.with_columns([
    pl.col(c).cast(pl.Float64) for c in smart_cols
])

lazy_df.limit(10).collect()

1. Connecting to S3 to find files...
2. Skimming metadata to build the Master Schema...
Overriding problematic data types in the Master Schema...
Success! Discovered 197 unique columns across the timeline.

3. Scanning the entire dataset with the unified schema...


date,serial_number,model,capacity_bytes,failure,smart_1_raw,smart_5_raw,smart_9_raw,smart_194_raw,smart_197_raw,smart_1_normalized,smart_2_normalized,smart_2_raw,smart_3_normalized,smart_3_raw,smart_4_normalized,smart_4_raw,smart_5_normalized,smart_7_normalized,smart_7_raw,smart_8_normalized,smart_8_raw,smart_9_normalized,smart_10_normalized,smart_10_raw,smart_12_normalized,smart_12_raw,smart_184_normalized,smart_184_raw,smart_187_normalized,smart_187_raw,smart_188_normalized,smart_188_raw,smart_189_normalized,smart_189_raw,smart_190_normalized,smart_190_raw,…,smart_166_raw,smart_167_normalized,smart_167_raw,smart_169_normalized,smart_169_raw,smart_176_normalized,smart_176_raw,smart_178_normalized,smart_178_raw,smart_171_normalized,smart_171_raw,smart_172_normalized,smart_172_raw,smart_230_normalized,smart_230_raw,smart_244_normalized,smart_244_raw,smart_246_normalized,smart_246_raw,vault_id,pod_id,is_legacy_format,smart_71_normalized,smart_71_raw,smart_90_normalized,smart_90_raw,datacenter,cluster_id,pod_slot_num,smart_82_normalized,smart_82_raw,smart_27_normalized,smart_27_raw,smart_211_normalized,smart_211_raw,smart_212_normalized,smart_212_raw
date,str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i8,i8,f64,f64,f64,f64,str,str,i8,f64,f64,f64,f64,f64,f64,f64,f64
2013-04-10,"""MJ0351YNG9Z0XA""","""Hitachi HDS5C3030ALA630""",3000592982016,0,0.0,0.0,4031.0,26.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2013-04-10,"""MJ0351YNG9WJSA""","""Hitachi HDS5C3030ALA630""",3000592982016,0,0.0,2.0,4099.0,29.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2013-04-10,"""MJ0351YNG9Z7LA""","""Hitachi HDS5C3030ALA630""",3000592982016,0,0.0,0.0,3593.0,26.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2013-04-10,"""MJ0351YNGAD37A""","""Hitachi HDS5C3030ALA630""",3000592982016,0,0.0,0.0,2339.0,29.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2013-04-10,"""MJ0351YNGABYAA""","""Hitachi HDS5C3030ALA630""",3000592982016,0,0.0,0.0,2741.0,25.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2013-04-10,"""MJ1311YNG7ESHA""","""Hitachi HDS5C3030ALA630""",3000592982016,0,0.0,0.0,8723.0,20.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null

In [4]:
bucket = "backblaze-afr"
prefix = "processed/drivestats/data"

year_pattern = "2025"
path = f"s3://{bucket}/{prefix}/date_month={year_pattern}-*/*.parquet"

print(f"Scanning data for {year_pattern}...")

essential_cols = [
    "date", "serial_number", "model", "failure",
    "smart_1_raw", "smart_10_raw", "smart_188_raw",
    "smart_5_raw", "smart_187_raw", "smart_197_raw"
]
smart_cols = [c for c in essential_cols if "smart" in c]

lazy_df = pl.scan_parquet(
    path,
    storage_options={"region": "us-west-1"},
    schema=master_schema,
    missing_columns="insert",
    extra_columns="ignore",
    cast_options=pl.ScanCastOptions(integer_cast='upcast')
)
pruned_lazy_df = lazy_df.select(essential_cols).with_columns([
    pl.col(c).cast(pl.Float32) for c in smart_cols
])

Scanning data for 2025...


In [5]:
os.makedirs("data_ready", exist_ok=True)
local_save_path = "data_ready/df_2025_pruned.parquet"

print("Saving ML-ready dataset to local SageMaker disk...")

pruned_lazy_df.sink_parquet(local_save_path)

print(f"Success! File saved at: {local_save_path}")

Saving ML-ready dataset to local SageMaker disk...
Success! File saved at: data_ready/df_2025_pruned.parquet


In [6]:
print("Scanning for empty columns...")
null_check = pl.scan_parquet(
    path,
    storage_options={"region": "us-west-1"},
    schema=master_schema,
    missing_columns="insert",
    extra_columns="ignore",
    cast_options=pl.ScanCastOptions(integer_cast='upcast')
).select([
    (pl.col(c).is_null().sum() == pl.len()).alias(c) 
    for c in master_schema.keys()
]).collect(streaming=True) 

valid_cols = [col for col in null_check.columns if not null_check[col][0]]
print(f"Out of {len(master_schema)} columns, only {len(valid_cols)} have actual data.")

os.makedirs("data_ready", exist_ok=True)
local_save_path = "data_ready/df_2025_valid_only.parquet"

smart_cols_to_cast = [c for c in valid_cols if "smart" in c]

print(f"Streaming the {len(valid_cols)} useful columns to disk...")

lazy_df_full = pl.scan_parquet(
    path,
    storage_options={"region": "us-west-1"},
    schema=master_schema,
    missing_columns="insert",
    extra_columns="ignore",
    cast_options=pl.ScanCastOptions(integer_cast='upcast')
).select(valid_cols).with_columns([
    pl.col(c).cast(pl.Float32) for c in smart_cols_to_cast
])

lazy_df_full.sink_parquet(local_save_path)

print(f"Success! Optimized dataset saved to: {local_save_path}")

Scanning for empty columns...


/tmp/ipykernel_8028/125845714.py:12: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  ]).collect(streaming=True)


Out of 197 columns, only 177 have actual data.
Streaming the 177 useful columns to disk...
Success! Optimized dataset saved to: data_ready/df_2025_valid_only.parquet
